# **Building a Reflection Agent with External Knowledge Integration**


Estimated time needed: **30** minutes


In this lab, you will build a deep research agent that uses a technique called **Reflection**. This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive. We will be building an agent that acts as a nutritional expert, capable of providing detailed, evidence-based advice.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#Writing-the-Code">Writing the Code</a>
        <ol>
            <li><a href="#Tavily-Search-API-Key-Setup">Tavily Search API Key Setup</a></li>
            <li><a href="#Tool-Setup:-Tavily-Search">Tool Setup: Tavily Search</a></li>
            <li><a href="#LLM-and-Prompting">LLM and Prompting</a></li>
            <li><a href="#Defining-the-Responder">Defining the Responder</a></li>
            <li><a href="#Tool-Execution">Tool Execution</a></li>
            <li><a href="#Defining-the-Revisor">Defining the Revisor</a></li>
        </ol>
    </li>
    <li><a href="#Building-the-Graph">Building the Graph</a></li>
    <li><a href="#Running-the-Agent">Running the Agent</a></li>
</ol>


## Objectives

After completing this lab, you will be able to:

 - Understand the core principles of the Reflexion framework.
 - Build an agent that can critique and improve its own responses.
 - Use LangGraph to create a cyclical, iterative agent workflow.
 - Integrate external tools, such as web search, into a LangChain agent.
 - Construct complex prompts for nuanced agent behavior.


----


## Setup


For this lab, we will be using the following libraries:

* [`langgraph`](https://pypi.org/project/langgraph/) for stateful graph workflows.
* [`langchain[openai]`](https://pypi.org/project/langchain/) for LangChain's current OpenAI chat model interface.
* [`langchain-tavily`](https://pypi.org/project/langchain-tavily/) for Tavily web search tools.
* [`python-dotenv`](https://pypi.org/project/python-dotenv/) for loading `OPENAI_API_KEY`, `OPENAI_MODEL`, and `TAVILY_API_KEY` from `.env`.


### Installing Required Libraries
Run the following to install the required libraries (it might take a few minutes):


In [ ]:
%%capture
%pip install -q -U "langgraph" "langchain[openai]" "langchain-tavily" "python-dotenv"


### Importing Required Libraries



In [ ]:
import json
import os
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, MessagesState, StateGraph
from pydantic import BaseModel, Field


# API Configuration

This lab uses OpenAI for language-model calls and Tavily for web search. Store keys in a local `.env` file instead of hard-coding them in the notebook.

Example `.env` file:

```text
OPENAI_API_KEY=your_openai_api_key_here
OPENAI_MODEL=gpt-5-nano
TAVILY_API_KEY=your_tavily_api_key_here
```

`OPENAI_MODEL` is optional. If it is not set, the notebook uses `gpt-5-nano`.


In [ ]:
# The setup cells below call load_dotenv() and read API keys from .env.
# Keep secrets in .env. Do not paste API keys directly into notebook cells.


---


## Writing the Code


### API Key Setup

This lab needs:

- `OPENAI_API_KEY` for the chat model
- `TAVILY_API_KEY` for web search

Both are loaded from `.env` with `python-dotenv`.


In [ ]:
load_dotenv()

missing_keys = [
    key for key in ("OPENAI_API_KEY", "TAVILY_API_KEY")
    if not os.getenv(key)
]

if missing_keys:
    raise ValueError(
        "Missing required environment variable(s): " + ", ".join(missing_keys)
    )


### Tool Setup: Tavily Search

Our agent needs a search tool to gather external evidence. We use the current `langchain_tavily.TavilySearch` integration, which reads `TAVILY_API_KEY` from the environment.


In [ ]:
tavily_tool = TavilySearch(max_results=3, topic="general")

sample_query = "healthy breakfast recipes for blood sugar control"
search_results = tavily_tool.invoke(sample_query)
print(search_results)


### LLM and Prompting

At the core of the agent is an OpenAI chat model initialized through LangChain's current `init_chat_model` interface.


In [ ]:
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
llm = init_chat_model(OPENAI_MODEL, model_provider="openai")

question = "Any ideas for a healthy breakfast?"
response = llm.invoke(question).content
print(response)


In [ ]:
# Reuse the same sample question for the structured Reflexion chain.
question


#### Crafting the Agent's Personas and Logic

The Reflexion workflow uses two personas:

- **Responder**: drafts an answer, critiques it, and proposes search queries.
- **Revisor**: uses tool results to revise the answer and include citations.


In [ ]:
responder_system_prompt = """\
You are a careful evidence-oriented research assistant.

Your job:
- answer the user's question with the best information you currently have
- critique your own answer
- identify missing information and unsupported claims
- propose search queries that would verify or improve the answer

Do not pretend to have external evidence yet. If evidence is needed, add
search queries. Keep the draft answer concise, practical, and cautious.
"""

revisor_system_prompt = """\
You are a rigorous answer revisor.

Your job:
- use the previous draft, self-critique, and Tavily search results
- revise the answer so it is more accurate and better supported
- remove unsupported or superfluous claims
- include citations as URLs when tool results provide them
- propose more search queries only if another revision would materially help

Prefer clear, actionable, evidence-backed answers. For health topics, avoid
medical diagnosis and recommend professional medical advice where appropriate.
"""

responder_prompt = ChatPromptTemplate.from_messages([
    ("system", responder_system_prompt),
    MessagesPlaceholder(variable_name="messages"),
])

revisor_prompt = ChatPromptTemplate.from_messages([
    ("system", revisor_system_prompt),
    MessagesPlaceholder(variable_name="messages"),
])


### Defining the Responder

The **Responder** is the first component of the agent's thinking process. It generates an initial draft using the responder persona defined above.

Here, we create a basic chain without structured output first, so we can compare plain text generation with the structured tool-call version in the next step.


In [ ]:
temp_chain = responder_prompt | llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)


#### Structuring the Agent's Output: Data Models

To make the agent's self-critique process reliable, we need to enforce a specific output structure. We use Pydantic `BaseModel` to define two data classes:

1.  `Reflection`: This class structures the self-critique, requiring the agent to identify what information is `missing` and what is `superfluous` (unnecessary).
2.  `AnswerQuestion`: This class structures the entire response. It forces the agent to provide its main `answer`, a `reflection` (using the `Reflection` class), and a list of `search_queries`.


In [ ]:
class Reflection(BaseModel):
    missing: str = Field(description="Important missing information")
    superfluous: str = Field(description="Unnecessary or unsupported information")

class AnswerQuestion(BaseModel):
    answer: str = Field(description="Draft answer to the user's question")
    reflection: Reflection = Field(description="Self-critique of the answer")
    search_queries: list[str] = Field(
        description="Search queries for external verification"
    )


#### Binding Tools to the Responder

Now, we bind the `AnswerQuestion` data model as a **tool** to our LLM chain. This crucial step forces the LLM to generate its output in the exact JSON format defined by our Pydantic classes. The LLM doesn't just write text; it calls this "tool" to structure its entire thought process.

After invoking this new chain, we can see the structured output, including the initial answer, the self-critique, and the generated search queries:


In [ ]:
initial_chain = responder_prompt | llm.bind_tools([AnswerQuestion])

response = initial_chain.invoke({
    "messages": [HumanMessage(content=question)]
})

print("--- Full Structured Output ---")
print(response.tool_calls)


In [ ]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

In [ ]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

In [ ]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

### Tool Execution

Now that the Responder has generated search queries based on its self-critique, the next step is to actually *execute* those searches. We'll define a function, `execute_tools`, that takes the agent's state, extracts the search queries, runs them through the Tavily tool, and returns the results.

We will also manage the conversation history in `response_list`:


In [ ]:
response_state = {
    "messages": [
        HumanMessage(content=question),
        response,
    ]
}


In [ ]:
tool_call = response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)


In [ ]:
def execute_tools(state: MessagesState) -> dict:
    last_ai_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_ai_message.tool_calls:
        search_queries = tool_call["args"].get("search_queries", [])
        query_results = {}

        for query in search_queries:
            query_results[query] = tavily_tool.invoke(query)

        tool_messages.append(
            ToolMessage(
                content=json.dumps(query_results, default=str),
                tool_call_id=tool_call["id"],
                name=tool_call["name"],
            )
        )

    return {"messages": tool_messages}


In [ ]:
tool_response = execute_tools(response_state)
response_state["messages"].extend(tool_response["messages"])


In [ ]:
tool_response

In [ ]:
response_state


### Defining the Revisor

The **Revisor** is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

We create a new set of instructions (`revise_instructions`) that guide the Revisor. These instructions emphasize:
- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.


In [ ]:
# The revisor persona is defined above in revisor_system_prompt.
# It asks the model to revise with evidence, remove unsupported claims,
# and include citations when available.


#### Structuring the Revisor's Output

Just as we did with the Responder, we define a Pydantic class, `ReviseAnswer`, to structure the Revisor's output. This class inherits from `AnswerQuestion` but adds a new field for `references`, ensuring the agent includes citations in its revised answer.

We then bind this new tool to the revisor chain:


In [ ]:
class ReviseAnswer(AnswerQuestion):
    """Revise the original answer using tool results."""

    citations: list[str] = Field(description="URLs supporting the revised answer")

revisor_chain = revisor_prompt | llm.bind_tools([ReviseAnswer])


#### Invoking the Revisor

Finally, we invoke the `revisor_chain`, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.


In [ ]:
response = revisor_chain.invoke({"messages": response_state["messages"]})
print("--- Revised Answer with Citations ---")
print(response.tool_calls[0]["args"])


In [ ]:
response_state["messages"].append(response)


## Building the Graph

Now we will use **LangGraph** to assemble these components—Responder, Tool Executor, and Revisor—into a cohesive, cyclical workflow. A graph is a natural way to represent this process, where nodes represent the different stages of thinking and edges represent the flow of information between them.

### Defining the Event Loop

The core of our graph is the event loop. This function determines whether the agent should continue its revision process or if it has reached a satisfactory conclusion. We'll set a maximum number of iterations to prevent the agent from getting stuck in an infinite loop:


In [ ]:
MAX_ITERATIONS = 4


In [ ]:
def event_loop(state: MessagesState) -> Literal["execute_tools", END]:
    tool_visits = sum(
        isinstance(message, ToolMessage)
        for message in state["messages"]
    )

    if tool_visits >= MAX_ITERATIONS:
        return END
    return "execute_tools"


In [ ]:
graph = StateGraph(MessagesState)

graph.add_node("respond", lambda state: {
    "messages": [initial_chain.invoke({"messages": state["messages"]})]
})
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", lambda state: {
    "messages": [revisor_chain.invoke({"messages": state["messages"]})]
})


In [ ]:
graph.add_edge(START, "respond")
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")


In [ ]:
graph.add_conditional_edges("revisor", event_loop)


## Running the Agent

With our graph compiled, we're ready to run the full Reflection agent. We'll give it a new, more complex query that requires careful, evidence-based advice.

As the agent runs, we can see the entire process unfold: the initial draft, the self-critique, the tool searches, and the final, revised answer that incorporates the new evidence.


In [ ]:
app = graph.compile()

inputs = {
    "messages": [
        HumanMessage(
            content=(
                "I'm pre-diabetic and need to lower my blood sugar, and I have heart issues. "
                "What breakfast foods should I eat and avoid?"
            )
        )
    ]
}

responses = app.invoke(inputs)


In [ ]:
print("--- Initial Draft Answer ---")
initial_answer = responses["messages"][1].tool_calls[0]["args"]["answer"]
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

for msg in reversed(responses["messages"]):
    if getattr(msg, "tool_calls", None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get("args", {}).get("answer")
            if answer:
                answers.append(answer)

for i, answer in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{answer}\n")


#### **Plotting the Graph**

Visualize the compiled LangGraph workflow as Mermaid.


In [ ]:
from IPython.display import Markdown, display

mermaid = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid}\n```"))


## Authors


[Joseph Santarcangelo](https://author.skills.network/instructors/joseph_santarcangelo)


[Faranak Heidari](https://author.skills.network/instructors/faranak_heidari)


### Other Contributors


[Abdul Fatir](https://author.skills.network/instructors/abdul_fatir)


## Change Log


<details>
    <summary>Click here for the changelog</summary>


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2025-06-24|0.5|Leah Hanson|QA review and grammar fixes|
|2025-06-24|0.4|Steve Ryan|ID review and format/typo fixes|
|2025-06-16|0.3|Abdul Fatir|Updated Lab|
|2025-06-10|0.2|Joseph Santarcangelo|Changed Project Architecture|
|2025-05-30|0.1|Faranak Heidari and Joseph Santarcangelo |Created Lab|

</details>


Copyright © IBM Corporation. All rights reserved.
